# Module 3: Leakage-Safe TF-IDF Logistic-Regression Baseline

**Question:** How strong is a transparent sparse-text baseline when its feature fitting,
model selection and test access follow the same evidence controls planned for RoBERTa?

**Success criteria**

- fit every TF-IDF vocabulary on training messages only;
- select a predeclared candidate using validation macro-F1 only;
- unlock the official test split only after selection is hash-locked;
- publish results without persisting message text;
- treat confidence as uncalibrated until the calibration module.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    path
    for path in [current, *current.parents]
    if (path / 'pyproject.toml').exists()
)
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT


## Registered plan

The candidate sweep was written to YAML before fitting. All candidates use the same seed,
training rows and one CPU thread. Character features are included because short banking
requests often contain spelling variation and informative word fragments.

Selection order: macro-F1, accuracy, lower log-loss, then candidate name.


In [ ]:
from governed_banking.baseline import (
    BaselineConfig,
    validate_evaluation_artifact,
    validate_selection_artifact,
)
from governed_banking.data import sha256_file, stable_json_sha256, validate_manifest

BASELINE_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'baseline_tfidf.yaml'
DATASET_MANIFEST_PATH = (
    PROJECT_ROOT / 'data' / 'manifests' / 'banking77-seed-42.json'
)
SELECTION_PATH = (
    PROJECT_ROOT / 'reports' / 'baseline' / 'tfidf-logreg-selection.json'
)
EVALUATION_PATH = PROJECT_ROOT / 'reports' / 'baseline' / 'tfidf-logreg-test.json'
PREDICTIONS_PATH = (
    PROJECT_ROOT / 'reports' / 'baseline' / 'tfidf-logreg-test-predictions.jsonl'
)

config = BaselineConfig.from_yaml(BASELINE_CONFIG_PATH)
pd.DataFrame(
    [
        {
            'candidate': candidate.name,
            'word_ngrams': candidate.word.ngram_range,
            'char_ngrams': None if candidate.char is None else candidate.char.ngram_range,
            'C': candidate.c_value,
        }
        for candidate in config.candidates
    ]
)


## Validate the selection lock

This cell checks the current dataset manifest, experiment configuration and implementation
hashes against the selection report. A code or configuration change invalidates the lock.


In [ ]:
dataset_manifest = json.loads(DATASET_MANIFEST_PATH.read_text(encoding='utf-8'))
selection = json.loads(SELECTION_PATH.read_text(encoding='utf-8'))
validate_manifest(dataset_manifest)

implementation_sha256 = {
    'baseline.py': sha256_file(PROJECT_ROOT / 'src/governed_banking/baseline.py'),
    'data.py': sha256_file(PROJECT_ROOT / 'src/governed_banking/data.py'),
    'run_tfidf_baseline.py': sha256_file(
        PROJECT_ROOT / 'scripts/run_tfidf_baseline.py'
    ),
}
validate_selection_artifact(
    selection,
    dataset_manifest_sha256=dataset_manifest['manifest_sha256'],
    config_sha256=sha256_file(BASELINE_CONFIG_PATH),
    implementation_sha256=implementation_sha256,
)
selection['data_boundary']


In [ ]:
candidate_results = pd.DataFrame(
    [
        {
            'rank': result['validation_rank'],
            'candidate': result['candidate_name'],
            'features': result['feature_count'],
            'macro_f1': result['metrics']['macro_f1'],
            'accuracy': result['metrics']['accuracy'],
            'log_loss': result['metrics']['log_loss'],
            'fit_seconds': result['fit_seconds'],
            'converged': result['converged'],
        }
        for result in selection['candidate_results']
    ]
).sort_values('rank')
candidate_results


In [ ]:
ax = candidate_results.plot.bar(
    x='candidate',
    y='macro_f1',
    legend=False,
    color=['#5b8c3a', '#8abf62', '#d0643b'],
    ylim=(0.84, 0.91),
    figsize=(8, 4),
)
ax.set_title('Validation macro-F1 by registered candidate')
ax.set_ylabel('Macro-F1')
ax.set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()


## Verify the locked-test evidence

The report is valid only if it points to the unchanged selection lock. The prediction
artifact contains source indices, labels and uncalibrated confidence—but no messages.


In [ ]:
evaluation = json.loads(EVALUATION_PATH.read_text(encoding='utf-8'))
validate_evaluation_artifact(
    evaluation,
    selection_sha256=selection['selection_sha256'],
    dataset_manifest_sha256=dataset_manifest['manifest_sha256'],
    config_sha256=sha256_file(BASELINE_CONFIG_PATH),
    implementation_sha256=implementation_sha256,
)
prediction_rows = [
    json.loads(line)
    for line in PREDICTIONS_PATH.read_text(encoding='utf-8').splitlines()
]
assert len(prediction_rows) == 3_080
assert stable_json_sha256(prediction_rows) == evaluation['test_predictions_sha256']
assert all('text' not in row and 'message' not in row for row in prediction_rows)
print('Selection, evaluation and prediction evidence validated.')


In [ ]:
test_metrics = evaluation['test_result']['metrics']
pd.Series(
    {
        'accuracy': test_metrics['accuracy'],
        'macro_f1': test_metrics['macro_f1'],
        'weighted_f1': test_metrics['weighted_f1'],
        'log_loss': test_metrics['log_loss'],
        'top_3_accuracy': test_metrics['top_3_accuracy'],
        'mean_max_confidence_uncalibrated': (
            test_metrics['mean_max_confidence_uncalibrated']
        ),
    },
    name='locked_test',
)


## Failure analysis

Aggregate accuracy can conceal operationally important errors. These tables identify
low-performing intents and recurring directional confusions.


In [ ]:
worst_intents = (
    pd.DataFrame.from_dict(test_metrics['per_intent'], orient='index')
    .rename_axis('intent')
    .sort_values(['f1', 'recall'])
    .head(10)
)
worst_intents


In [ ]:
pd.DataFrame(test_metrics['top_confusions']).head(10)


## Interpretation and next decision

- The selected word + character model is a credible baseline, not a toy comparison.
- Test macro-F1 is **0.9053** across all 77 intents.
- Transfer-state, top-up-state and identity-verification boundaries remain weak.
- Maximum probability is not a routing threshold; these probabilities are not calibrated.
- Module 4 must beat or complement this baseline under the same manifest contract.
- Later policy work should evaluate specific confusion costs, not average accuracy alone.

To reproduce deliberately, run python scripts/run_tfidf_baseline.py run from the repository
root. This creates a new timing-dependent evidence hash and replaces the local model file.
